# Day 5 – Final Regression – Healthcare Model Evaluation

This notebook keeps the healthcare `age_band` analysis and adds a simple ML model evaluation similar to the reference `ml_flow` notebook.

The model predicts `patient_status` from patient attributes in `dim_patient` and reports accuracy, precision, recall and F1.

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from pyspark.sql import functions as F
from mlflow.models import infer_signature
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

CATALOG = "capstone_8hospital"
GOLD = f"{CATALOG}.gold_schema"
SEED = 42

print("Healthcare model evaluation started")

In [0]:
patients = spark.table(f"{GOLD}.dim_patient")

patient_age = (patients
    .groupBy("age_band")
    .agg(F.countDistinct("patient_key").alias("patient_count"))
    .orderBy("age_band"))

display(patient_age)

In [0]:
from pyspark.sql import functions as F

encounters = spark.table(f"{GOLD}.fact_patient_encounter")

encounter_age = (encounters
    .join(patients.select("patient_key", "age_band"), "patient_key", "left")
    .groupBy("age_band")
    .agg(F.countDistinct("encounter_key").alias("encounter_count"))
    .orderBy("age_band"))

display(encounter_age)

## Model setup

In [0]:
pdf = patients.select(
    "age_band", "sex", "city", "insurance_type", "primary_language", "patient_status"
).dropna(subset=["patient_status"]).toPandas()

FEATURES = ["age_band", "sex", "city", "insurance_type", "primary_language"]
TARGET = "patient_status"

print("Rows:", len(pdf))
print("Classes:", sorted(pdf[TARGET].astype(str).unique().tolist()))

In [0]:
train, test = train_test_split(
    pdf, test_size=0.20, random_state=SEED, stratify=pdf[TARGET]
)

NUM = []
CAT = FEATURES

def make_pipeline(model):
    prep = ColumnTransformer([
        ("cat", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CAT)
    ])
    return Pipeline([("prep", prep), ("model", model)])

models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2,
        class_weight="balanced", random_state=SEED, n_jobs=-1
    )
}

results, pipes = [], {}

for name, model in models.items():
    pipe = make_pipeline(model).fit(train[FEATURES], train[TARGET])
    pred = pipe.predict(test[FEATURES])
    scores = {
        "test_accuracy": metrics.accuracy_score(test[TARGET], pred),
        "test_precision": metrics.precision_score(test[TARGET], pred, average="weighted", zero_division=0),
        "test_recall": metrics.recall_score(test[TARGET], pred, average="weighted", zero_division=0),
        "test_f1": metrics.f1_score(test[TARGET], pred, average="weighted", zero_division=0)
    }
    with mlflow.start_run(run_name=name):
        mlflow.log_params({"model": name, "seed": SEED, "n_train": len(train), "n_test": len(test)})
        mlflow.log_metrics(scores)
    pipes[name] = pipe
    results.append({"model": name, **scores})

results = pd.DataFrame(results).sort_values("test_accuracy", ascending=False)
display(results.round(4))
best_name = results.iloc[0]["model"]
print("Best model:", best_name)

## Accuracy and model evaluation

In [0]:
best = pipes[best_name]
y_pred = best.predict(test[FEATURES])

test_scores = {
    "accuracy": metrics.accuracy_score(test[TARGET], y_pred),
    "precision": metrics.precision_score(test[TARGET], y_pred, average="weighted", zero_division=0),
    "recall": metrics.recall_score(test[TARGET], y_pred, average="weighted", zero_division=0),
    "f1": metrics.f1_score(test[TARGET], y_pred, average="weighted", zero_division=0)
}

display(pd.DataFrame([test_scores]).round(4))
print("Model accuracy:", round(test_scores["accuracy"], 4))

fig, ax = plt.subplots(figsize=(6, 5))
metrics.ConfusionMatrixDisplay.from_predictions(
    test[TARGET], y_pred, ax=ax, colorbar=False
)
plt.tight_layout()
plt.show()

In [0]:
signature = infer_signature(
    test[FEATURES].head(20),
    best.predict(test[FEATURES].head(20))
)

with mlflow.start_run(run_name=f"final_{best_name}"):
    mlflow.log_params({
        "model": best_name,
        "seed": SEED,
        "n_train": len(train),
        "n_test": len(test)
    })
    mlflow.log_metrics({
        "accuracy": test_scores["accuracy"],
        "precision": test_scores["precision"],
        "recall": test_scores["recall"],
        "f1": test_scores["f1"]
    })
    model_info = mlflow.sklearn.log_model(
        best, name="model", signature=signature,
        input_example=test[FEATURES].head(3)
    )

print("Model saved:", model_info.model_uri)

## Final regression status

In [0]:
failed = 0
warned = 0

if test_scores["accuracy"] >= 0.70:
    print("[PASS] Model accuracy >= 0.70")
else:
    print("[WARN] Model accuracy < 0.70")
    warned += 1

final_status = "FAILED" if failed > 0 else "SUCCESS"
print("WARN:", warned)
print("FAIL:", failed)
print("Final status:", final_status)

assert failed == 0, "Final regression failed"